<a href="https://colab.research.google.com/github/richayanamandra/GenAI-Experiments/blob/main/genAI_lab4_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load the Dataset
text_data = """
artificial intelligence is transforming modern society.
it is used in healthcare finance education and transportation.
machine learning allows systems to improve automatically with experience.
data plays a critical role in training intelligent systems.
large datasets help models learn complex patterns.
deep learning uses multi layer neural networks.
neural networks are inspired by biological neurons.
each neuron processes input and produces an output.
training a neural network requires optimization techniques.
gradient descent minimizes the loss function.
natural language processing helps computers understand human language.
text generation is a key task in nlp.
language models predict the next word or character.
recurrent neural networks handle sequential data.
lstm and gru models address long term dependency problems.
however rnn based models are slow for long sequences.
transformer models changed the field of nlp.
they rely on self attention mechanisms.
attention allows the model to focus on relevant context.
transformers process data in parallel.
this makes training faster and more efficient.
modern language models are based on transformers.
education is being improved using artificial intelligence.
intelligent tutoring systems personalize learning.
automated grading saves time for teachers.
online education platforms use recommendation systems.
technology enhances the quality of learning experiences.
ethical considerations are important in artificial intelligence.
fairness transparency and accountability must be ensured.
ai systems should be designed responsibly.
data privacy and security are major concerns.
researchers continue to improve ai safety.
text generation models can create stories poems and articles.
they are used in chatbots virtual assistants and content creation.
generated text should be meaningful and coherent.
evaluation of text generation is challenging.
human judgement is often required.
continuous learning is essential in the field of ai.
research and innovation drive technological progress.
students should build strong foundations in mathematics.
programming skills are important for ai engineers.
practical experimentation enhances understanding.
"""
print("Libraries imported and dataset loaded.")

Libraries imported and dataset loaded.


In [2]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences (n-grams)
input_sequences = []
for line in text_data.strip().split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Split into predictors and labels
X, labels = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(labels, num_classes=total_words)

print(f"Total Words in Vocab: {total_words}")
print(f"Max Sequence Length: {max_sequence_len}")
print("Data preprocessed and ready for the model.")

Total Words in Vocab: 195
Max Sequence Length: 10
Data preprocessed and ready for the model.


In [3]:
# Design the LSTM Architecture
model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=64, input_length=max_sequence_len-1))

# Stacked LSTMs for deeper context understanding
model.add(LSTM(100, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))

model.add(Dense(total_words, activation='softmax'))

# Compile the model
optimizer = Adam(learning_rate=0.005)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Train the Model
print("Starting LSTM training...")
# 100 epochs is highly optimal for this architecture and dataset size
model.fit(X, y, epochs=100, verbose=1, batch_size=16)
print("Training complete!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Starting LSTM training...
Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.0077 - loss: 5.2690
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0447 - loss: 5.0454
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0476 - loss: 4.8871
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0479 - loss: 4.7538
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0217 - loss: 4.7040
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0431 - loss: 4.4662
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0620 - loss: 4.3148
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0814 - loss: 4.0450
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1317 - loss: 3.7667
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1213 - loss: 3.6230
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1978 - loss: 3.2939
Epoch 12/100
16/16 ━━━━━━━━━━━━━━━━━━

In [5]:
def generate_text_with_temperature(seed_text, next_words, model, max_sequence_len, temperature=1.0):
    for _ in range(next_words):
        # Convert seed to sequence and pad
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Predict probabilities for the next word
        predicted_probs = model.predict(token_list, verbose=0)[0]

        # Apply temperature scaling
        predicted_probs = np.asarray(predicted_probs).astype('float64')
        predicted_probs = np.log(predicted_probs + 1e-7) / temperature
        exp_preds = np.exp(predicted_probs)
        predicted_probs = exp_preds / np.sum(exp_preds)

        # Sample the next word based on the adjusted probabilities
        probas = np.random.multinomial(1, predicted_probs, 1)
        predicted_index = np.argmax(probas)

        # Map index back to word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

words_to_generate = 12
# Adjust this between 0.1 (strict) and 1.5 (creative)
temp_setting = 0.8

seeds = [
    "artificial intelligence",
    "data plays",
    "text generation models",
    "ethical considerations",
    "machine learning"
]

for seed in seeds:
    generated = generate_text_with_temperature(seed, words_to_generate, model, max_sequence_len, temperature=temp_setting)
    print(f"Seed: '{seed}'\nOutput: '{generated}'\n")

Seed: 'artificial intelligence'
Output: 'artificial intelligence is transforming modern society society and transportation output neurons creation problems problems'

Seed: 'data plays'
Output: 'data plays a critical role in training intelligent systems systems systems of with language'

Seed: 'text generation models'
Output: 'text generation models can create stories poems and articles output problems problems content creation creation'

Seed: 'ethical considerations'
Output: 'ethical considerations are important in artificial intelligence mathematics education education education intelligence using artificial'

Seed: 'machine learning'
Output: 'machine learning allows systems to improve automatically with experience context context problems problems problems'

